# RSNA Knee — Kaggle runner

Thin wrapper. All logic lives in the repo's `src/`, which arrives here as an
attached Kaggle Dataset imported from GitHub.

**Order matters:** refresh the dataset on Kaggle *before* re-running, or this
executes the previous revision. Cell 2 prints the revision so a stale run is
obvious instead of silent.

Set `SCRIPT` in the last cell to pick the job:

| `SCRIPT`  | what it does                                   | Internet | Accelerator |
|-----------|------------------------------------------------|----------|-------------|
| `extract` | report → 12 labels, one time over 4407 reports  | **on**   | GPU T4 x2   |
| `eval`    | grade those labels against the 58 gold studies  | off      | CPU is fine |
| `train`   | image model → `submission.csv`                  | off      | GPU T4 x2   |

`extract` must run *before* `eval`, and `eval` reads the CSV from
`/kaggle/working`, which does not survive the session — so run them together,
or save the output (see the last cell).

Use T4, not P100 — Kaggle's PyTorch ships without Pascal kernels.


In [ ]:
# Locate the code dataset. Kaggle may nest the repo one level down inside the
# dataset root, so probe both levels. Deliberately shallow: a recursive walk
# would crawl every DICOM in the competition data.
import os

CODE = None
for entry in sorted(os.listdir("/kaggle/input")):
    base = f"/kaggle/input/{entry}"
    if not os.path.isdir(base):
        continue
    candidates = [base] + [f"{base}/{x}" for x in sorted(os.listdir(base))]
    for path in candidates:
        if os.path.exists(f"{path}/src/train.py"):
            CODE = path

print("CODE =", CODE)
assert CODE, "no dataset under /kaggle/input contains src/train.py"

In [ ]:
# Staleness check: does this match the VERSION you just pushed?
print(open(f"{CODE}/VERSION").read())

In [ ]:
# Pick the job, then run it. Each entry point reads its own config from the repo.
SCRIPT = "extract"  # "extract", "train", or "eval"

import runpy, sys

if SCRIPT == "extract":
    # One-time report -> label job. Needs Internet ON for the model weights.
    # --limit 20 is a smoke test: prove the weights download and the 12-bit
    # answer parses, before spending hours on all 4407. Raise the limit in steps.
    argv = ["extract_labels.py",
            "--config", f"{CODE}/configs/extract.yaml",
            "--limit", "20"]
elif SCRIPT == "eval":
    # Scores labels_derived.csv against the 58 gold studies. Reads the CSV from
    # the work dir by default, so it only works in the session that made it.
    argv = ["eval_extractor.py", "--config", f"{CODE}/configs/extract.yaml"]
elif SCRIPT == "train":
    argv = ["train.py", "--config", f"{CODE}/configs/base.yaml"]
else:
    raise SystemExit(f"unknown SCRIPT {SCRIPT!r} - use 'extract', 'eval' or 'train'")

print("running", *argv)
sys.argv = argv
runpy.run_path(f"{CODE}/src/{argv[0]}", run_name="__main__")


## `extract` — the one-time label job

1. Internet **on**, accelerator T4 x2.
2. Run the smoke test (`--limit 20`) and read the printed
   `[parse] exact=… loose=… fail=…` line and the s/report timing before
   committing to the full run. A `fail` count above zero is worth inspecting.
3. Raise `--limit` in steps (20 → 300 → all). `/kaggle/working` does **not**
   survive a session, so after each run use **Save Version → Save & Run All**,
   or copy `labels_derived.csv` into a Kaggle Dataset. `resume: true` skips
   studies already in the file, so a killed session continues rather than
   restarts — but only if the file survives, which is the whole point of saving.
4. Delete `labels_derived.csv` before the final full run if the smoke test
   used a different model or config, so stale rows cannot hide in the output.

## `train` — after the debug run is clean

1. Set `debug: false` in `configs/base.yaml` (or pass `--no-debug`).
2. Bump `VERSION`, push, refresh the Kaggle dataset.
3. Confirm the printed marker changed, then run full.

Full runs need checkpointing — a session is capped at ~12h and gets killed
at the limit, so never leave the only copy of a trained model in memory.
